# Notebook 03 — Análisis Exploratorio de Datos (EDA)
## Proyecto Integrador | Minería de Datos I
**Integrante:** Val Martinetti

---
### Preguntas de análisis que guían este notebook

1. ¿Cuál es el perfil típico del usuario de la plataforma?
2. ¿El tiempo de visualización varía significativamente según el plan de suscripción?
3. ¿Existe relación entre edad y consumo de contenido?
4. ¿Cómo se distribuyen los usuarios por país y género favorito?
5. ¿El nivel de soporte técnico requerido varía según el plan?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuración visual
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

# Cargar dataset limpio
df = pd.read_csv('../data/processed/streaming_users_clean.csv',
                 parse_dates=['last_login_date'])

print(f"Dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")
df.head()

## 1. Análisis Univariado

### 1.1 Distribución de la edad (`age`) — Variable cuantitativa continua

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
axes[0].hist(df['age'], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].axvline(df['age'].mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Media: {df["age"].mean():.1f}')
axes[0].axvline(df['age'].median(), color='orange', linestyle='--', linewidth=1.5, label=f'Mediana: {df["age"].median():.1f}')
axes[0].set_xlabel('Edad (años)')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Distribución de edad — Histograma')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['age'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[1].set_xlabel('Edad (años)')
axes[1].set_title('Distribución de edad — Boxplot')

plt.tight_layout()
plt.savefig('../reports/fig01_age_distribucion.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Media:   {df['age'].mean():.1f} años")
print(f"Mediana: {df['age'].median():.1f} años")
print(f"Desvío:  {df['age'].std():.1f} años")
print(f"IQR:     {df['age'].quantile(0.75) - df['age'].quantile(0.25):.1f} años")
print(f"Asimetría (skewness): {df['age'].skew():.3f}")

**Interpretación:** la distribución de edad es aproximadamente simétrica (skewness ≈ 0.14, muy cercana a 0). Media y mediana son similares (~33 años), lo que indica ausencia de outliers relevantes luego de la limpieza. El 50% central de los usuarios tiene entre 25 y 42 años. La plataforma tiene una base de usuarios relativamente joven, con presencia en todas las franjas etarias adultas.

### 1.2 Distribución de `monthly_watch_time_mins` — Variable cuantitativa continua

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histograma
axes[0].hist(df['monthly_watch_time_mins'], bins=40, color='teal', edgecolor='white', alpha=0.8)
axes[0].axvline(df['monthly_watch_time_mins'].mean(),   color='red',    linestyle='--', lw=1.5, label=f'Media: {df["monthly_watch_time_mins"].mean():.0f}')
axes[0].axvline(df['monthly_watch_time_mins'].median(), color='orange', linestyle='--', lw=1.5, label=f'Mediana: {df["monthly_watch_time_mins"].median():.0f}')
axes[0].set_xlabel('Minutos/mes')
axes[0].set_ylabel('Frecuencia')
axes[0].set_title('Tiempo de visualización mensual')
axes[0].legend()

# Boxplot
axes[1].boxplot(df['monthly_watch_time_mins'].dropna(), vert=False, patch_artist=True,
                boxprops=dict(facecolor='teal', alpha=0.6))
axes[1].set_xlabel('Minutos/mes')
axes[1].set_title('Tiempo de visualización — Boxplot')

plt.tight_layout()
plt.savefig('../reports/fig02_watch_time_dist.png', dpi=100, bbox_inches='tight')
plt.show()

desc = df['monthly_watch_time_mins'].describe()
print(desc.round(1))
print(f"Asimetría: {df['monthly_watch_time_mins'].skew():.3f}")
print(f"IQR: {desc['75%'] - desc['25%']:.0f} min")

**Interpretación:** la distribución del tiempo de visualización tiene asimetría positiva moderada (media > mediana), con cola hacia valores altos que corresponde a heavy users. El 50% central oscila entre ~490 y ~1.046 minutos/mes (≈ 8–17 horas), lo cual representa un consumo moderado a intenso. La winsorización k=3 aplicada en la limpieza eliminó el valor centinela 99.999, sin afectar el patrón real de los heavy users.

### 1.3 Distribución de `subscription_plan` — Variable cualitativa

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
counts = df['subscription_plan'].value_counts()
bars = ax.bar(counts.index, counts.values, color=['#4e79a7', '#f28e2b', '#59a14f'], alpha=0.85)
ax.bar_label(bars, fmt='%d', padding=4, fontsize=10)
ax.set_title('Distribución de planes de suscripción')
ax.set_xlabel('Plan')
ax.set_ylabel('Cantidad de usuarios')
ax.set_ylim(0, counts.max() * 1.15)
plt.tight_layout()
plt.savefig('../reports/fig03_plan_distribucion.png', dpi=100, bbox_inches='tight')
plt.show()

print(counts)
print()
print((counts / len(df) * 100).round(1))

**Interpretación:** el plan **Básico** concentra el 44.9% de los usuarios, seguido por Estándar (35.3%) y Premium (19.8%). La distribución no es uniforme: casi la mitad de la base elige la opción más económica. Esto es relevante para decisiones de pricing y estrategia de contenido.

## 2. Análisis Bivariado

### 2.1 Tiempo de visualización por plan de suscripción

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Boxplot por plan
orden_plan = ['Básico', 'Estándar', 'Premium']
df.boxplot(column='monthly_watch_time_mins', by='subscription_plan', ax=axes[0],
           positions=range(len(orden_plan)), notch=False, patch_artist=True)
axes[0].set_xticklabels(orden_plan)
axes[0].set_title('Watch time por plan')
axes[0].set_xlabel('Plan de suscripción')
axes[0].set_ylabel('Minutos/mes')
plt.sca(axes[0]); plt.title('Watch time por plan'); plt.suptitle('')

# Medias por plan
medias = df.groupby('subscription_plan')['monthly_watch_time_mins'].mean().reindex(orden_plan)
colors = ['#4e79a7', '#f28e2b', '#59a14f']
bars = axes[1].bar(medias.index, medias.values, color=colors, alpha=0.85)
axes[1].bar_label(bars, fmt='%.0f min', padding=4)
axes[1].set_title('Tiempo promedio de visualización por plan')
axes[1].set_ylabel('Minutos/mes (promedio)')
axes[1].set_ylim(0, medias.max() * 1.2)

plt.tight_layout()
plt.savefig('../reports/fig04_watch_por_plan.png', dpi=100, bbox_inches='tight')
plt.show()

for plan in orden_plan:
    g = df[df['subscription_plan'] == plan]['monthly_watch_time_mins']
    print(f"{plan}: media={g.mean():.0f} min | mediana={g.median():.0f} min | n={len(g)}")

**Interpretación:** se confirma la hipótesis de análisis: los usuarios Premium visualizan significativamente más contenido (media ~1.300 min/mes) que los Estándar (~750 min) y Básico (~540 min). La brecha es notable y consistente. Esto refleja que el plan Premium atrae a heavy users o que el mayor acceso a contenido incentiva mayor consumo. La relación plan→consumo podría ser bidireccional y merece investigación causal adicional.

### 2.2 Relación entre edad y tiempo de visualización

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

scatter = ax.scatter(
    df['age'], df['monthly_watch_time_mins'],
    alpha=0.15, s=10, color='steelblue'
)

# Línea de tendencia
z = np.polyfit(df['age'].dropna(), 
               df.loc[df['age'].notna(), 'monthly_watch_time_mins'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['age'].min(), df['age'].max(), 100)
ax.plot(x_line, p(x_line), 'r--', linewidth=2, label=f'Tendencia (pendiente: {z[0]:.1f} min/año)')

corr = df[['age', 'monthly_watch_time_mins']].corr().iloc[0,1]
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Minutos visualizados/mes')
ax.set_title(f'Edad vs. Tiempo de visualización (r = {corr:.3f})')
ax.legend()

plt.tight_layout()
plt.savefig('../reports/fig05_age_vs_watch.png', dpi=100, bbox_inches='tight')
plt.show()

print(f"Correlación de Pearson: {corr:.4f}")

**Interpretación:** la correlación entre edad y tiempo de visualización es prácticamente nula (r ≈ 0.00), y la línea de tendencia es casi horizontal. Esto indica que la edad **no predice** el consumo de contenido: usuarios jóvenes y mayores consumen cantidades similares de contenido. El plan de suscripción (analizado en 2.1) es un predictor mucho más relevante que la edad.

### 2.3 Distribución de géneros favoritos por plan

In [ ]:
pivot = pd.crosstab(df['subscription_plan'], df['favorite_genre'], normalize='index') * 100
pivot = pivot.reindex(orden_plan)

fig, ax = plt.subplots(figsize=(12, 5))
pivot.T.plot(kind='bar', ax=ax, colormap='Set2', alpha=0.85, edgecolor='white')
ax.set_xlabel('Género favorito')
ax.set_ylabel('% dentro del plan')
ax.set_title('Preferencia de género por plan de suscripción (%)')
ax.legend(title='Plan')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../reports/fig06_genre_por_plan.png', dpi=100, bbox_inches='tight')
plt.show()

print(pivot.round(1))

**Interpretación:** la distribución de géneros es relativamente uniforme entre planes, sin diferencias marcadas. Todos los planes muestran preferencias similares entre Comedia, Drama, Romance y Thriller. El género Documental tiene una ligera mayor proporción en Premium (~15%) vs. Básico (~14%), aunque la diferencia es pequeña. Esto sugiere que el contenido preferido no determina la elección de plan.

### 2.4 Heatmap de correlaciones entre variables numéricas

In [ ]:
numeric_cols = ['age', 'monthly_watch_time_mins', 'customer_support_tickets']
corr_matrix = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0,
            fmt='.3f', square=True, ax=ax, linewidths=0.5)
ax.set_title('Matriz de correlación — Variables numéricas')
plt.tight_layout()
plt.savefig('../reports/fig07_heatmap_corr.png', dpi=100, bbox_inches='tight')
plt.show()

print(corr_matrix.round(4))

**Interpretación:** las tres variables numéricas presentan correlaciones cercanas a 0 entre sí. Esto indica que son **prácticamente independientes**: la edad no predice el tiempo de visualización ni los tickets de soporte, y el consumo de contenido no se relaciona linealmente con la necesidad de soporte técnico. Desde la perspectiva del PCA (notebook 04), la ausencia de correlación implica que las variables ya capturan dimensiones de variación distintas.

## 3. Análisis Multivariado

### 3.1 Tiempo de visualización, edad y plan — scatter con hue

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Muestra para no saturar la visualización
muestra = df.sample(1500, random_state=42)

fig, ax = plt.subplots(figsize=(10, 6))
palette = {'Básico': '#4e79a7', 'Estándar': '#f28e2b', 'Premium': '#59a14f'}
for plan, grupo in muestra.groupby('subscription_plan'):
    ax.scatter(grupo['age'], grupo['monthly_watch_time_mins'],
               label=plan, color=palette[plan], alpha=0.4, s=20)

ax.set_xlabel('Edad (años)')
ax.set_ylabel('Minutos visualizados/mes')
ax.set_title('Edad vs. Watch time según plan de suscripción')
ax.legend(title='Plan')
plt.tight_layout()
plt.savefig('../reports/fig08_scatter_multivariado.png', dpi=100, bbox_inches='tight')
plt.show()

**Interpretación:** la visualización multivariada revela que las tres nubes (una por plan) se superponen horizontalmente en todo el rango etario, confirmando que la edad no determina el plan. Sin embargo, las nubes se **separan verticalmente**: los puntos verdes (Premium) se concentran en la parte superior del eje Y, mientras los azules (Básico) en la inferior. Esto consolida el hallazgo del análisis bivariado: el plan es el principal factor diferenciador del comportamiento de consumo, independientemente de la edad del usuario.

## Síntesis de hallazgos del EDA

In [ ]:
print("""
HALLAZGOS PRINCIPALES DEL EDA
══════════════════════════════════════════════════════════════

1. PERFIL TÍPICO DEL USUARIO:
   - Edad: ~33 años (distribución simétrica, rango 6–100)
   - Plan más frecuente: Básico (45% de la base)
   - Visualización: ~750 min/mes en promedio (varía mucho por plan)
   - Géneros: distribución uniforme entre 7 géneros

2. EL PLAN ES EL PRINCIPAL DIFERENCIADOR DE CONSUMO:
   - Premium: ~1.300 min/mes (heavy users)
   - Estándar: ~750 min/mes
   - Básico: ~540 min/mes
   La brecha Premium–Básico es del +140%

3. LA EDAD NO PREDICE EL CONSUMO:
   - Correlación edad–watch_time ≈ 0.00
   - Usuarios de todas las edades pueden ser heavy o light users

4. VARIABLES NUMÉRICAS INDEPENDIENTES ENTRE SÍ:
   - Todas las correlaciones < 0.05 en valor absoluto
   - Cada variable captura una dimensión diferente del usuario

5. DISTRIBUCIÓN GEOGRÁFICA EQUITATIVA:
   - 7 países con ~1.150 usuarios cada uno (distribución uniforme)
   - No se detecta concentración geográfica dominante

Para el PCA: las 3 variables numéricas (age, watch_time, tickets)
son buenas candidatas al ser independientes entre sí y cubrir
dimensiones distintas del comportamiento del usuario.
""")